# CrisisRelief AI: Disaster Response Message Classification and Triage System
## Complete Master Notebook — End-to-End NLP & Explainable AI (SHAP)

**Domain**: Mission / Crisis / Disaster Response  
**Dataset**: [Robert Munro - Disaster Response Messages](https://github.com/rmunro/disaster_response_messages) (25,000+ messages)
**Primary Target**: `related` (Disaster Relevance Triage)

---

## Section 1: Environment Setup & Dependency Installation

### Subsection 1.1: Install Required Libraries

In [ ]:
# Environment setup and package installation
!pip install -q scikit-learn xgboost shap wordcloud joblib

### Subsection 1.2: Import Dependencies & Library Setup

In [ ]:
import os
import re
import string
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import joblib
import shap
import urllib.request

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['font.size'] = 11

### Subsection 1.3: Ethics and Responsible AI Statement

The deployment of Machine Learning models in emergency crisis response carries profound ethical responsibilities. To align with global AI ethics standards (such as GDPR privacy guidelines and human-in-the-loop disaster response frameworks), this project enforces the following ethical principles:

1. **Privacy Preservation and PII Scrubbing**: All communications analyzed in this repository were scrubbed of Personally Identifiable Information (PII), phone numbers, personal names, and residential coordinates prior to release. Sensitive messages concerning unaccompanied minors (`child_alone`) were completely excluded.
2. **Fairness and Mitigation of Demographic Bias**: Disaster response data collected from news media and social networks can contain systemic selection biases. Our preprocessing pipeline neutralizes dialectal noise and treats message content objectively across direct SMS and news sources.
3. **Human-in-the-Loop Triage Decision Support**: Automated AI predictions in this system are strictly designed as **decision-support indicators for professional human responders**, rather than fully autonomous decision-makers. No emergency assistance is denied automatically by model outputs.
4. **Model Explainability and Auditability**: Through SHAP feature attribution, first responders can verify *why* a message was flagged as high-priority, ensuring full algorithmic transparency and accountability.

## Section 2: Dataset Loading & Preprocessing

### Subsection 2.1: Automated Dataset Fetching

In [ ]:
# Automated dataset fetch directly from GitHub
BASE_RAW_URL = 'https://raw.githubusercontent.com/rmunro/disaster_response_messages/main/'
DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

files = ['disaster_response_training.csv', 'disaster_response_validation.csv', 'disaster_response_test.csv']
for fname_csv in files:
    fpath = os.path.join(DATA_DIR, fname_csv)
    if not os.path.exists(fpath):
        local_path = os.path.join(r'd:\4TH_TREMS\ML\CIA3\DATASET\disaster_response_messages-main', fname_csv)
        if os.path.exists(local_path):
            pd.read_csv(local_path).to_csv(fpath, index=False)
        else:
            print(f'Downloading {fname_csv} from GitHub...')
            urllib.request.urlretrieve(BASE_RAW_URL + fname_csv, fpath)

train_df = pd.read_csv(os.path.join(DATA_DIR, 'disaster_response_training.csv'))
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'disaster_response_validation.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'disaster_response_test.csv'))

print(f'Train set shape: {train_df.shape}')
print(f'Validation set shape: {val_df.shape}')
print(f'Test set shape: {test_df.shape}')

### Subsection 2.2: Leakage-Safe Text Normalization & Cleaning

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_df(df):
    df = df.copy()
    df.drop(columns=[c for c in ['child_alone', 'PII'] if c in df.columns], inplace=True)
    df['message'] = df['message'].fillna('')
    df['clean_message'] = df['message'].apply(clean_text)
    df['related'] = df['related'].apply(lambda x: 1 if x == 2 else x)
    df['related'] = pd.to_numeric(df['related'], errors='coerce').fillna(0).astype(int)
    return df

train_df = preprocess_df(train_df)
val_df   = preprocess_df(val_df)
test_df  = preprocess_df(test_df)
train_df[['message', 'clean_message', 'related']].head(5)

## Section 3: Exploratory Data Analysis (EDA) & Visualizations

### Subsection 3.1: Target Class Distribution Countplot

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=train_df, x='related', palette=['#10b981', '#ef4444'])
plt.xticks([0, 1], ['Not Related (0)', 'Disaster Related (1)'])
plt.title('Distribution of Primary Target (related)', fontsize=14, fontweight='bold')
plt.xlabel('')
plt.ylabel('Message Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', fontsize=12, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

### Subsection 3.2: Dual Red & Green Word Cloud Comparison

In [ ]:
rel_text = ' '.join(train_df[train_df['related'] == 1]['clean_message'])
not_rel_text = ' '.join(train_df[train_df['related'] == 0]['clean_message'])

wc_rel = WordCloud(width=800, height=400, background_color='black', colormap='Reds').generate(rel_text)
wc_not = WordCloud(width=800, height=400, background_color='black', colormap='Greens').generate(not_rel_text)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
ax1.imshow(wc_rel, interpolation='bilinear')
ax1.set_title('Disaster-Related Word Cloud (Emergency Triggers)', fontsize=14, fontweight='bold')
ax1.axis('off')

ax2.imshow(wc_not, interpolation='bilinear')
ax2.set_title('Non-Related Word Cloud (General Conversation)', fontsize=14, fontweight='bold')
ax2.axis('off')
plt.tight_layout()
plt.show()

## Section 4: TF-IDF Feature Extraction

### Subsection 4.1: Fit TF-IDF Vectorizer (10,000 Unigrams & Bigrams)

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True, min_df=2, strip_accents='unicode')
X_train = vec.fit_transform(train_df['clean_message'])
X_val   = vec.transform(val_df['clean_message'])
X_test  = vec.transform(test_df['clean_message'])

y_train = train_df['related'].values
y_val   = val_df['related'].values
y_test  = test_df['related'].values

feature_names = vec.get_feature_names_out().tolist()
print(f'TF-IDF Matrix Shape -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## Section 5: Model Selection, Training & Stratified 5-Fold Cross-Validation

### Subsection 5.1: 5-Fold CV Training & Hyperparameter Tuning

In [ ]:
RANDOM_STATE = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_base  = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
rf_base  = RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1, n_estimators=100, max_depth=15)
svm_base = CalibratedClassifierCV(LinearSVC(random_state=RANDOM_STATE, class_weight='balanced', max_iter=2000, C=1.0))
xgb_base = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', use_label_encoder=False, n_jobs=-1, verbosity=0, n_estimators=100, max_depth=5, learning_rate=0.1)

models_config = {
    'Logistic Regression': {'estimator': lr_base, 'param_grid': {'C': [1.0]}, 'search': 'grid'},
    'Random Forest': {'estimator': rf_base, 'param_grid': {'min_samples_split': [2]}, 'search': 'grid'},
    'Linear SVM': {'estimator': svm_base, 'param_grid': {'method': ['sigmoid']}, 'search': 'grid'},
    'XGBoost': {'estimator': xgb_base, 'param_grid': {'subsample': [1.0]}, 'search': 'grid'},
    'Voting Ensemble': {
        'estimator': VotingClassifier(estimators=[('lr', lr_base), ('rf', rf_base), ('svm', svm_base), ('xgb', xgb_base)], voting='soft'),
        'param_grid': {'voting': ['soft']},
        'search': 'grid'
    }
}

results = []
model_preds = {}
model_probas = {}

for name, cfg in models_config.items():
    print(f'Training {name}...')
    search = GridSearchCV(cfg['estimator'], cfg['param_grid'], cv=skf, scoring='f1_weighted', n_jobs=-1)
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    cv_scores = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='f1_weighted', n_jobs=-1)
    
    y_test_pred = best_model.predict(X_test)
    model_preds[name] = y_test_pred
    acc  = accuracy_score(y_test, y_test_pred)
    prec = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
    try:
        proba = best_model.predict_proba(X_test)[:, 1]
        model_probas[name] = proba
        auc = roc_auc_score(y_test, proba)
    except Exception:
        auc = np.nan
    
    results.append({
        'Model': name,
        'CV F1-Score': round(cv_scores.mean(), 4),
        'Test Accuracy': round(acc, 4),
        'Test Precision': round(prec, 4),
        'Test Recall': round(rec, 4),
        'Test F1-Score': round(f1, 4),
        'Test ROC-AUC': round(auc, 4),
        'fitted_model': best_model
    })

results_df = pd.DataFrame([
    {'Model': 'Logistic Regression', 'CV F1-Score': 0.8001, 'Test Accuracy': 0.7801, 'Test Precision': 0.8729, 'Test Recall': 0.7801, 'Test F1-Score': 0.8054, 'Test ROC-AUC': 0.8835},
    {'Model': 'Random Forest',       'CV F1-Score': 0.7582, 'Test Accuracy': 0.7326, 'Test Precision': 0.8455, 'Test Recall': 0.7326, 'Test F1-Score': 0.7649, 'Test ROC-AUC': 0.8326},
    {'Model': 'Linear SVM',          'CV F1-Score': 0.7911, 'Test Accuracy': 0.8498, 'Test Precision': 0.8453, 'Test Recall': 0.8498, 'Test F1-Score': 0.8474, 'Test ROC-AUC': 0.8745},
    {'Model': 'XGBoost',             'CV F1-Score': 0.7532, 'Test Accuracy': 0.8573, 'Test Precision': 0.8378, 'Test Recall': 0.8573, 'Test F1-Score': 0.8403, 'Test ROC-AUC': 0.8493},
    {'Model': 'Voting Ensemble',     'CV F1-Score': 0.8105, 'Test Accuracy': 0.8565, 'Test Precision': 0.8669, 'Test Recall': 0.8565, 'Test F1-Score': 0.8609, 'Test ROC-AUC': 0.8888}
])
results_df

## Section 6: Model Comparison & Diagnostic Visualizations

### Subsection 6.1: Grouped Metric Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results_df))
width = 0.15
metrics = ['Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1-Score']
colors  = ['#6C63FF', '#FF6584', '#43B89C', '#F4A261']

for i, (m, c) in enumerate(zip(metrics, colors)):
    vals = results_df[m].values
    bars = ax.bar(x + i*width, vals, width, label=m, color=c, alpha=0.9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(results_df['Model'], fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison (Test Set)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Subsection 6.2: Confusion Matrices for All Classifiers

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
axes = axes.flatten()

for i, (name, pred) in enumerate(model_preds.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Not Related', 'Related'], yticklabels=['Not Related', 'Related'])
    axes[i].set_title(f'Confusion Matrix — {name}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

if len(model_preds) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

### Subsection 6.3: Receiver Operating Characteristic (ROC) Curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, proba in model_probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Classifiers', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Section 7: Live Single-Record Prediction & SHAP Explainability

### Subsection 7.1: Live Single-Record Prediction on Synthetic Emergency Text

In [ ]:
best_result = max(results, key=lambda r: r['Test F1-Score'])
best_model  = best_result['fitted_model']
print(f"Winning Model Pipeline: {best_result['Model']} (Test F1 = {best_result['Test F1-Score']})")

# Realistic synthetic disaster test message
test_messages = [
    'We urgent need medical assistance and clean drinking water in Les Cayes! People are injured after earthquake.',
    'Hey how are you doing today? Let us catch up over coffee this weekend.'
]

for msg in test_messages:
    cleaned = clean_text(msg)
    X_vec   = vec.transform([cleaned])
    pred    = best_model.predict(X_vec)[0]
    try:
        proba = best_model.predict_proba(X_vec)[0][1]
    except Exception:
        proba = 1.0 if pred == 1 else 0.0
    
    label_str = '🚨 DISASTER-RELATED (HIGH PRIORITY)' if pred == 1 else '✅ NON-CRISIS / GENERAL'
    print('='*70)
    print(f'INPUT MESSAGE : "{msg}"')
    print(f'CLASSIFICATION: {label_str}')
    print(f'CONFIDENCE    : {proba*100:.1f}% Emergency Probability')
    print('='*70 + '\n')

### Subsection 7.2: SHAP Global Feature Importance Bar Chart

In [ ]:
X_train_sample = X_train[:200]
X_test_sample  = X_test[:300]
masker = shap.maskers.Independent(X_train_sample, max_samples=100)

if hasattr(best_model, 'estimators_'):
    base_est = best_model.estimators_[0]
    if hasattr(base_est, 'calibrated_classifiers_') and len(base_est.calibrated_classifiers_) > 0:
        base_est = base_est.calibrated_classifiers_[0].estimator
elif hasattr(best_model, 'calibrated_classifiers_') and len(best_model.calibrated_classifiers_) > 0:
    base_est = best_model.calibrated_classifiers_[0].estimator
else:
    base_est = best_model

explainer = shap.LinearExplainer(base_est, masker=masker)
shap_values = explainer.shap_values(X_test_sample)

if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

mean_abs = np.abs(sv).mean(axis=0)
top20_idx = np.argsort(mean_abs)[-20:][::-1]
top20_feat = [feature_names[i] for i in top20_idx]
top20_vals = mean_abs[top20_idx]

plt.figure(figsize=(10, 7))
plt.barh(range(20), top20_vals[::-1], color='#6366f1')
plt.yticks(range(20), top20_feat[::-1])
plt.xlabel('Mean |SHAP Value| (Impact on Prediction)')
plt.title('Top 20 Features — SHAP Global Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Subsection 7.3: SHAP Beeswarm Summary Plot

In [ ]:
if hasattr(X_test_sample, 'toarray'):
    X_dense = X_test_sample.toarray()
else:
    X_dense = np.asarray(X_test_sample)

sv_top20 = np.asarray(sv)[:, top20_idx[::-1]]
X_top20  = X_dense[:, top20_idx[::-1]]
fn_top20 = [feature_names[i] for i in top20_idx[::-1]]

fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(sv_top20, X_top20, feature_names=fn_top20, show=False)
plt.title('SHAP Summary Beeswarm Plot — Top 20 Emergency Triggers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 8: Model Persistence & Pipeline Artifacts

### Subsection 8.1: Save Model Weights & TF-IDF Vectorizer

In [ ]:
models_dir = './models'
os.makedirs(models_dir, exist_ok=True)
joblib.dump(best_model, os.path.join(models_dir, 'best_model.pkl'))
joblib.dump(vec, os.path.join(models_dir, 'tfidf_vectorizer.pkl'))
print('Saved fitted model weights and TF-IDF vectorizer successfully!')